# 02. Data Preparation, Feature Engineering & WOE/IV
## Financial Analytics and Risk Reporting System

**Objective:** Clean raw loan and ledger datasets, handle missing values and outliers, engineer credit features, calculate Weight of Evidence (WOE) and Information Value (IV), and perform feature selection.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")
from src.config import get_data_path
from src.data_cleaning import clean_loan_data, clean_ledger_data
from src.feature_engineering import create_loan_features, calculate_all_woe_iv

print("Modules imported successfully.")


### 1. Ingest Raw Datasets


In [ ]:
raw_dir = get_data_path("raw")
df_loans_raw = pd.read_csv(os.path.join(raw_dir, "loans_raw.csv"))
df_ledger_raw = pd.read_csv(os.path.join(raw_dir, "ledger_raw.csv"))
print(f"Raw loans: {df_loans_raw.shape}, Raw ledger: {df_ledger_raw.shape}")


### 2. Execute Data Cleaning Pipeline


In [ ]:
df_loans_clean = clean_loan_data(df_loans_raw)
df_ledger_clean = clean_ledger_data(df_ledger_raw)
print("Cleaned shapes:", df_loans_clean.shape, df_ledger_clean.shape)


### 3. Feature Engineering


In [ ]:
df_loans_feat = create_loan_features(df_loans_clean)
print("Derived feature columns:", [c for c in df_loans_feat.columns if c not in df_loans_clean.columns])


### 4. Weight of Evidence (WOE) and Information Value (IV) Analysis


In [ ]:
df_model = df_loans_feat.dropna(subset=["default_flag"]).copy()
candidate_features = ["dti", "annual_inc", "loan_amnt", "int_rate", "revol_util", "inq_last_6mths"]

summary_iv, details = calculate_all_woe_iv(df_model, candidate_features, target="default_flag", n_bins=5)
print("Feature Information Value (IV) Summary:")
print(summary_iv)


### 5. WOE Binning Tables for Top Predictors


In [ ]:
for feat in summary_iv["feature"].head(3):
    print(f"\n=== WOE Table for {feat} (IV = {details[feat]["iv"]}) ===")
    print(details[feat]["woe_table"][["bin", "n", "events", "non_events", "woe", "iv_contribution"]])


### 6. Persist Staging Datasets


In [ ]:
stg_dir = get_data_path("staging")
os.makedirs(stg_dir, exist_ok=True)
df_loans_feat.to_csv(os.path.join(stg_dir, "stg_loans.csv"), index=False)
df_ledger_clean.to_csv(os.path.join(stg_dir, "stg_ledger.csv"), index=False)
print("Staged datasets saved successfully.")
